In [1]:
!pip install mysql-connector-python

In [7]:
import mysql.connector

print("MySQL Library Ready")

MySQL Library Ready


In [53]:
import mysql.connector
import json
import requests

In [107]:
api_key="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"
model="meta-llama/llama-3.1-8b-instruct"

connection=mysql.connector.connect(
    host="localhost",
    user="root",
    password="Dhanu@123",
    database="banking_ai"
    
)

print("mysql connection successful")

def tool_fetch_sql_balance(account_id):
    cursor=connection.cursor()
    query="""select customer_name ,balance from customer_bal where account_id=%s;"""

    cursor.execute(query,(account_id,))
    result=cursor.fetchone()
    cursor.close()
    if result:
        return {
           "customer_name" : result[0],
           "balance" : result[1]
        }
    else :
        return "Account not found"
def tool_calling_flow(user_message):
    system_prompt="""
   Role:  You are an AI Routing Agent.

Task:
Your task is to identify the user's request and decide whether a tool should be called.
You must never answer banking information from your own knowledge.

Available Tool:

tool_fetch_sql_balance(account_id)

Description:
- This tool fetches the customer's name and account balance from the SQL database using the account ID.

Rules:

1. If the user asks about:
   - Bank balance
   - Account balance
   - Available balance
   - Amount in account

   ALWAYS call:
   tool_fetch_sql_balance(account_id)

2. Never generate or guess a balance yourself.

3. Never refuse a balance request if the account ID is provided.

4. If the account ID is available, extract it and pass it to the tool.

5. Return ONLY valid JSON.

6. Do not include explanations, markdown, or extra text.

Output Format:

{
  "tool": "tool_fetch_sql_balance",
  "arguments": {
    "account_id": "ACC014"
  }
}

Example 1:

User:
Check my balance for account ACC014

Output:

{
  "tool": "tool_fetch_sql_balance",
  "arguments": {
    "account_id": "ACC014"
  }
}

Example 2:

User:
What is the available balance for ACC010?

Output:

{
  "tool": "tool_fetch_sql_balance",
  "arguments": {
    "account_id": "ACC010"
  }
}

Example 3:

User:
Show my account balance for ACC021.

Output:

{
  "tool": "tool_fetch_sql_balance",
  "arguments": {
    "account_id": "ACC021"
  }
}
    
    
    """
    headers={
        "Authorization":f"Bearer {api_key}",
        "Content-Type":"application/json"
    }

    payload={
        "model":model,
        "messages":[{"role":"system", "content":system_prompt},
                    {"role":"user", "content":user_message}
                   ]
    }

    end_point="https://openrouter.ai/api/v1/chat/completions"

    response=requests.post(end_point,headers=headers,json=payload)
    result=response.json()["choices"][0]["message"]["content"]
    
    try:
        tool_call=json.loads(result)
        if ("tool" in tool_call and tool_call["tool"]=="tool_fetch_sql_balance"):
            account_id=tool_call["arguments"]["account_id"]
            tool_result=tool_fetch_sql_balance(account_id)
            if isinstance(tool_result,dict):
                return (
                f"customer_name: {tool_result['customer_name']}\n"
                f"account_id: {account_id}\n"   
                f"balance: {tool_result['balance']}"
                )
            else:
                return tool_result
        else :
            return result
    except:
        return result
            

 
fun=tool_calling_flow("check my account balance ACC014")
print(fun)

#func=tool_fetch_sql_balance("ACC014")
#print(f"customer_name: {func['customer_name']}")
#print(f"balance: {func['balance']}")

mysql connection successful
customer_name: Meera Iyer
account_id: ACC014
balance: 7400.00
